# New artifact

This notebook lives on the lab's own disk and is gone when the lab scales down. Declaring with `commit=True` keeps it: the last saved version is copied to the declared artifact's folder as `declaring.ipynb`. (This cell will be replaced with the artifact path.)

In [ ]:
import lab
from artifacts.core.artifact import Resources
from artifacts.core.SGD.lr_schedule import WarmupCosine
from artifacts.core.SGD.optimizers import AdamW
from artifacts.core.SGD.training import LoopConfig, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer
from artifacts.profiles import Profile
from profiling.profile_events import launch_chain
from profiling.util import run_profile

In [ ]:
mobydick = SourceURL(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
montecristo = SourceURL(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
romeojuliet = SourceURL(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")

tokenizer = BPETokenizer(
    vocab_size=10000,
    special_tokens=("<|endoftext|>",),
    sources=(mobydick, montecristo),
    allocated_resources=Resources(cpu=4),
)

dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(mobydick, montecristo),
    valid_sources=(romeojuliet,),
)

pretraining = Pretraining(
    run_id="my-run",
    dataset=dataset,
    model=Transformer(
        vocab_size=tokenizer.vocab_size,
        d_model=64,
        d_ff=128,
        sequence_length=32,
        num_layers=4,
        num_heads=16,
        rope_theta=10000,
        dtype="torch.float32",
    ),
    training_parameters=TrainingParameters(
        total_steps=20000,
        batch_size=64,
        max_norm=1.0,
        lr_schedule=WarmupCosine(
            max_learning_rate=3e-4,
            min_learning_rate=3e-5,
            warmup_iters=200,
            cosine_cycle_iters=5000,
        ),
        optimizer=AdamW(
            betas=(0.9, 0.95),
            weight_decay=0.1,
            eps=1e-8,
        ),
        seed=0,
    ),
    loop_config=LoopConfig(
        checkpoint_every=5000,
        val_every=200,
        gpu_check_every=200,
    ),
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
    tags={"experiment": ["my-experiment"]},
)

In [ ]:
pretraining.pprint()

## Declare

Preview first: it writes nothing. Save the notebook before committing, since the copy is the file on disk.

In [ ]:
report = lab.declare(pretraining, verbose=True)

In [ ]:
report = lab.declare(pretraining, verbose=True, commit=True)

## Benchmark

In [ ]:
result = run_profile(Profile(
    run_id=pretraining.run_id,
    resources=pretraining.allocated_resources,
    model=pretraining.model,
    optimizer=pretraining.training_parameters.optimizer,
    batch_size=pretraining.training_parameters.batch_size,
    seed=pretraining.training_parameters.seed,
    warmup_steps=5,
    profile_steps=4,
    autocast=None,  # PretrainJob trains without autocast
    kernel_events=True,
    module_scopes=True,
    peak_memory=True,
    trace_memory=True,
))

In [ ]:
result.display_steps(steps=[3])

In [ ]:
step = 3
rows = result.events(step)
groups, kernels, unattributed_kernels, flop_sources = result.aggregate_kernels(step, depth=3)
display(groups[["kernel_count", "gpu_time_us", "gpu_span_us", "total_est_flops"]])

In [ ]:
display(kernels[["name", "phase", "module_fqn", "owner_fqn", "gpu_start_us", "duration_us"]])

In [ ]:
kernel_id = kernels.index[0]
display(launch_chain(rows, kernel_id, format=True))